# 05 — Previsões com o modelo vigente

**Frente:** Dev 4 · **Critérios:** 6 (MLflow entre ciclos), 8 (modelo vigente em uso) e 9 (rastreabilidade)

Último notebook do ciclo, executado logo depois do `04_selecao_promocao`:

1. lê o manifesto `models/modelo_vigente.json` e carrega o modelo **pelo `run_id`** no MLflow;
2. prevê a partição de **teste**, que nunca participou de nenhuma escolha (votação, competição ou promoção);
3. mostra o que mudou entre os ciclos: `data_version`, features eleitas, modelo vigente e decisão;
4. registra as previsões num run `previsoes` do MLflow, com `cycle_id` e `data_version`.

O modelo salvo é um `Pipeline` completo (pré-processamento do Dev 1 → seleção das eleitas do Dev 2 → estimador do Dev 3), então recebe as colunas do parquet como estão, sem nenhum passo manual.

In [ ]:
import os, sys, json
from pathlib import Path

# funciona tanto rodando de notebooks/ (nbconvert) quanto da raiz
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(RAIZ)
sys.path.insert(0, str(RAIZ))
os.environ.setdefault("MLFLOW_DISABLE_AGENT_HINT", "1")
os.environ.setdefault("MLFLOW_ENABLE_ARTIFACTS_PROGRESS_BAR", "false")

import pandas as pd
import matplotlib.pyplot as plt

from src.ingestao import carregar_config, ler_registro
from src.preprocessamento import carregar_particoes, carregar_schema, ALVO
from src.rastreio import obter_cycle_id, iniciar_run
from src.previsoes import (carregar_vigente, prever, resumir, amostra, historico_ciclos,
                           diff_features, gravar_previsoes, limiar)

cfg = carregar_config()
CYCLE_ID = obter_cycle_id()
DATA_VERSION = carregar_schema(cfg)["data_version"]
print("cycle_id:    ", CYCLE_ID)
print("data_version:", DATA_VERSION)
pd.set_option("display.max_columns", 40); pd.set_option("display.width", 160)

## 1. Modelo vigente (pelo manifesto)

In [ ]:
manifesto, modelo = carregar_vigente(cfg)
TEM_VIGENTE = manifesto is not None

if not TEM_VIGENTE:
    print("Não há modelo vigente: nenhum candidato atingiu o F1 mínimo até agora. Nada a prever.")
else:
    if manifesto["cycle_id"] != CYCLE_ID and not CYCLE_ID.endswith("-local"):
        raise RuntimeError(f"o manifesto é do ciclo {manifesto['cycle_id']}, mas este é o {CYCLE_ID}: "
                           "o notebook 04 deste ciclo não gravou a decisão")
    campos = ["cycle_id", "decisao", "motivo", "run_id", "modelo", "algoritmo", "treinado_no_ciclo",
              "treinado_com_data_version", "promovido_no_ciclo", "n_features", "f1_validacao", "f1_teste"]
    display(pd.Series({k: manifesto.get(k) for k in campos}, name="manifesto").to_frame())
    print("features do vigente:", manifesto["features"])
    print()
    print(modelo)

## 2. Previsões no teste

O teste é a partição reservada por hash do `Address` (20%). Ele só aparece aqui e no número final do relatório. O limiar de classificação vem de `config.yaml` → `modelos.limiar_classificacao`.

In [ ]:
_, _, teste = carregar_particoes(cfg)
print(f"teste: {len(teste)} carteiras, {int(teste[ALVO].sum())} fraudes ({teste[ALVO].mean():.1%}) | limiar = {limiar(cfg)}")

if TEM_VIGENTE:
    previsoes = prever(modelo, teste, cfg)
    resumo = resumir(previsoes)
    display(pd.Series({k: (round(v, 4) if isinstance(v, float) else v) for k, v in resumo.items()},
                      name="teste").to_frame().T)
    if manifesto.get("f1_teste") is not None:
        # o 04 mediu o mesmo modelo na mesma partição: os números têm de bater
        assert abs(resumo["f1"] - manifesto["f1_teste"]) < 1e-9, "F1 de teste diferente do manifesto"
        print("F1 de teste confere com o manifesto do notebook 04 ✔")

Amostra estável (seed fixa), metade fraudes reais e metade legítimas, ordenada pela probabilidade:

In [ ]:
if TEM_VIGENTE:
    display(amostra(previsoes, n=20, seed=cfg["seeds"]["global"]).reset_index(drop=True)
            .style.format({"probabilidade_fraude": "{:.3f}"})
            .apply(lambda r: ["background-color: #fde2e2" if not r["acerto"] else "" for _ in r], axis=1))

In [ ]:
if TEM_VIGENTE:
    fig_prev, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
    for rot, g in previsoes.groupby("rotulo_real"):
        ax1.hist(g["probabilidade_fraude"], bins=40, alpha=0.6, density=True,
                 label="fraude" if rot == 1 else "legítima")
    ax1.axvline(limiar(cfg), ls="--", color="k", lw=1, label=f"limiar {limiar(cfg)}")
    ax1.set_xlabel("probabilidade de fraude"); ax1.set_title("Distribuição das probabilidades no teste"); ax1.legend()

    mc = pd.crosstab(previsoes["rotulo_real"], previsoes["previsao"]).reindex(index=[0, 1], columns=[0, 1], fill_value=0)
    ax2.imshow(mc.values, cmap="Blues")
    for i in range(2):
        for j in range(2):
            ax2.text(j, i, mc.values[i, j], ha="center", va="center", fontsize=13)
    ax2.set_xticks([0, 1], ["legítima", "fraude"]); ax2.set_yticks([0, 1], ["legítima", "fraude"])
    ax2.set_xlabel("previsto"); ax2.set_ylabel("real")
    ax2.set_title(f"Matriz de confusão — {manifesto['algoritmo']} (F1 = {resumo['f1']:.3f})")
    plt.tight_layout(); plt.show()

## 3. O que mudou entre os ciclos

Uma linha por ciclo decidido (`models/historico_decisoes.csv`, Dev 3), cruzada com os lotes ingeridos (`data/processed/lotes.json`, Dev 1) e as features eleitas (`artifacts/<cycle_id>/features_eleitas.json`, Dev 2).

In [ ]:
hist = historico_ciclos(cfg)
if hist.empty:
    print("ainda não há histórico de decisões")
else:
    cols = ["cycle_id", "data_version", "lotes_novos", "n_features", "vencedor_algoritmo", "f1_val_vencedor",
            "f1_val_vigente_anterior", "decisao", "vigente_algoritmo", "vigente_treinado_no_ciclo",
            "f1_val_vigente", "f1_teste_vigente", "mudou_data_version", "mudou_features", "mudou_modelo"]
    display(hist[[c for c in cols if c in hist]].round(4))

In [ ]:
if len(hist) >= 2:
    ant, atu = hist.iloc[-2], hist.iloc[-1]
    print(f"ciclo anterior: {ant['cycle_id']}  →  ciclo atual: {atu['cycle_id']}")
    print(f"  data_version: {ant['data_version']} → {atu['data_version']}"
          + ("" if atu["mudou_data_version"] else "  (sem dados novos)"))
    d = diff_features(ant["features_eleitas"], atu["features_eleitas"])
    print(f"  features: {len(d['mantidas'])} mantidas, entraram {d['entraram'] or '—'}, saíram {d['sairam'] or '—'}")
    print(f"  vigente:  {ant['vigente_algoritmo']} ({ant['vigente_run_id']}) → "
          f"{atu['vigente_algoritmo']} ({atu['vigente_run_id']})")
    print(f"  decisão:  {atu['decisao']} — {atu['motivo']}")
elif len(hist) == 1:
    print("primeiro ciclo: ainda não há ciclo anterior para comparar")

In [ ]:
lotes = pd.DataFrame(ler_registro(cfg)["lotes"])
lotes[["nome", "linhas", "fraudes", "cycle_id", "ingerido_em", "sha256"]].assign(sha256=lambda d: d["sha256"].str[:12])

## 4. Registro no MLflow e artefatos do ciclo

In [ ]:
if TEM_VIGENTE:
    caminhos = gravar_previsoes(previsoes, resumo, manifesto, CYCLE_ID, DATA_VERSION, cfg)
    with iniciar_run("previsoes", cycle_id=CYCLE_ID, data_version=DATA_VERSION, cfg=cfg,
                     vigente_run_id=manifesto["run_id"], algoritmo=manifesto["algoritmo"],
                     decisao=manifesto["decisao"]) as run:
        import mlflow
        mlflow.log_params({"vigente_run_id": manifesto["run_id"], "vigente_algoritmo": manifesto["algoritmo"],
                           "vigente_treinado_no_ciclo": manifesto["treinado_no_ciclo"],
                           "particao": "teste", "limiar": limiar(cfg)})
        mlflow.log_metrics({f"teste_{k}": v for k, v in resumo.items()})
        mlflow.log_artifact(str(caminhos["previsoes"]), "previsoes")
        mlflow.log_artifact(str(caminhos["resumo"]), "previsoes")
        mlflow.log_figure(fig_prev, "figuras/previsoes_teste.png")
        if not hist.empty:
            mlflow.log_text(hist.drop(columns=["features_eleitas"]).to_csv(index=False), "historico_ciclos.csv")
    print("run_id:", run.info.run_id)
    for k, p in caminhos.items():
        print(f"{k:10s}", p.relative_to(RAIZ).as_posix())
else:
    print("sem vigente: nenhum run de previsões neste ciclo")

### Para reproduzir fora do pipeline
```python
import mlflow, pandas as pd, json
mlflow.set_tracking_uri("sqlite:///mlflow.db")
man = json.load(open("models/modelo_vigente.json"))
modelo = mlflow.sklearn.load_model(f"runs:/{man['run_id']}/modelo")
proba = modelo.predict_proba(pd.read_parquet("data/processed/teste.parquet"))[:, 1]
```
Rodando de outra máquina, execute antes `python -m src.rastreio preparar` para ajustar os caminhos de artefato gravados no `mlflow.db`.